# 05. Evaluasi Model

Tahap evaluasi menguji dua hal. Pertama, kualitas retrieval: apakah ground truth masuk ke top-k hasil pencarian. Kedua, kualitas prediksi label reuse: apakah label yang diprediksi sesuai dengan label harapan pada `data/eval/queries.json`.

Metrik retrieval menggunakan `hit@k`, `top_1_correct`, `precision@k`, `recall@k`, dan `f1@k`. Karena setiap query memiliki satu ground truth utama, precision@5 bernilai 0,2 jika satu kasus relevan muncul dalam lima hasil teratas.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: /mnt/data/cbr_work/cbr-putusan-ite-pn-surabaya-main


In [2]:
import json
import pandas as pd
from IPython.display import display

from src.config import QUERIES_JSON, RETRIEVAL_METRICS_CSV, PREDICTION_METRICS_CSV
from src.evaluation import evaluate_retrieval, evaluate_prediction, build_failure_analysis

queries = json.loads(QUERIES_JSON.read_text(encoding="utf-8"))
print("Jumlah query evaluasi:", len(queries))
display(pd.DataFrame(queries))

Jumlah query evaluasi: 5


,query_id,query,ground_truth_case_id,expected_label
0,q1,terdakwa Maria Ulfa mendistribusikan informasi...,case_005,pidana penjara
1,q2,terdakwa Islah Jiwa Wilsman mengubah informasi...,case_007,bebas
2,q3,terdakwa Arjun Nadika Syahid mendistribusikan ...,case_006,pidana penjara
3,q4,terdakwa Muhammad Sobri menyiarkan informasi e...,case_010,pidana penjara
4,q5,terdakwa Syaifullah SE MSi bin Abd Hapi ancama...,case_034,bebas


In [3]:
retrieval_metrics = evaluate_retrieval(k=5)
prediction_metrics = evaluate_prediction(k=5)

print("Retrieval metrics disimpan di:", RETRIEVAL_METRICS_CSV)
print("Prediction metrics disimpan di:", PREDICTION_METRICS_CSV)

display(retrieval_metrics)
display(prediction_metrics)

Retrieval metrics disimpan di: /mnt/data/cbr_work/cbr-putusan-ite-pn-surabaya-main/data/eval/retrieval_metrics.csv
Prediction metrics disimpan di: /mnt/data/cbr_work/cbr-putusan-ite-pn-surabaya-main/data/eval/prediction_metrics.csv


,query_id,query,ground_truth_case_id,top_1_case_id,top_k_case_ids,hit_at_k,top_1_correct,precision_at_k,recall_at_k,f1_at_k
0,q1,terdakwa Maria Ulfa mendistribusikan informasi...,case_005,case_005,"case_005,case_017,case_028,case_026,case_008",1.0,1.0,0.2,1.0,0.333333
1,q2,terdakwa Islah Jiwa Wilsman mengubah informasi...,case_007,case_007,"case_007,case_045,case_050,case_046,case_027",1.0,1.0,0.2,1.0,0.333333
2,q3,terdakwa Arjun Nadika Syahid mendistribusikan ...,case_006,case_006,"case_006,case_009,case_048,case_021,case_015",1.0,1.0,0.2,1.0,0.333333
3,q4,terdakwa Muhammad Sobri menyiarkan informasi e...,case_010,case_010,"case_010,case_022,case_002,case_038,case_036",1.0,1.0,0.2,1.0,0.333333
4,q5,terdakwa Syaifullah SE MSi bin Abd Hapi ancama...,case_034,case_034,"case_034,case_012,case_022,case_049,case_032",1.0,1.0,0.2,1.0,0.333333
5,SUMMARY,,,,,1.0,1.0,0.2,1.0,0.333333


,query_id,query,expected_label,predicted_label,predicted_label_classifier,top_1_case_id,top_1_similarity,correct,precision_macro,recall_macro,f1_macro
0,q1,terdakwa Maria Ulfa mendistribusikan informasi...,pidana penjara,pidana penjara,pidana penjara,case_005,0.184065,1.0,NaN,NaN,NaN
1,q2,terdakwa Islah Jiwa Wilsman mengubah informasi...,bebas,bebas,pidana penjara,case_007,0.18267,1.0,NaN,NaN,NaN
2,q3,terdakwa Arjun Nadika Syahid mendistribusikan ...,pidana penjara,pidana penjara,pidana penjara,case_006,0.185196,1.0,NaN,NaN,NaN
3,q4,terdakwa Muhammad Sobri menyiarkan informasi e...,pidana penjara,pidana penjara,pidana penjara,case_010,0.100444,1.0,NaN,NaN,NaN
4,q5,terdakwa Syaifullah SE MSi bin Abd Hapi ancama...,bebas,bebas,pidana penjara,case_034,0.202406,1.0,NaN,NaN,NaN
5,SUMMARY,,,,,,,1.0,1.0,1.0,1.0


In [4]:
failure_analysis = build_failure_analysis(retrieval_metrics, prediction_metrics)
display(failure_analysis)

,query_id,failure_type,possible_cause,recommendation
0,q1,retrieval sesuai,"query terlalu pendek, variasi istilah hukum, m...","tambah query uji berlabel, bersihkan OCR/PDF l..."
1,q2,retrieval sesuai,"query terlalu pendek, variasi istilah hukum, m...","tambah query uji berlabel, bersihkan OCR/PDF l..."
2,q3,retrieval sesuai,"query terlalu pendek, variasi istilah hukum, m...","tambah query uji berlabel, bersihkan OCR/PDF l..."
3,q4,retrieval sesuai,"query terlalu pendek, variasi istilah hukum, m...","tambah query uji berlabel, bersihkan OCR/PDF l..."
4,q5,retrieval sesuai,"query terlalu pendek, variasi istilah hukum, m...","tambah query uji berlabel, bersihkan OCR/PDF l..."


## Analisis kegagalan dan rekomendasi perbaikan

Sumber kegagalan utama dalam CBR putusan pengadilan biasanya berasal dari empat hal. Pertama, kualitas ekstraksi PDF yang tidak stabil. Kedua, variasi istilah hukum dan variasi format putusan. Ketiga, ground truth yang belum cukup banyak. Keempat, query yang terlalu pendek sehingga konteks pasal, fakta, dan amar belum cukup jelas.

Rekomendasi perbaikan:

1. Tambah jumlah query evaluasi dengan label ground truth yang diverifikasi manual.
2. Perbaiki OCR atau ekstraksi teks untuk dokumen yang jumlah katanya sangat pendek.
3. Tambahkan bobot lebih tinggi untuk field `pasal`, `ringkasan_fakta`, dan `amar_putusan`.
4. Bandingkan baseline TF-IDF dengan embedding transformer seperti IndoBERT bila perangkat memungkinkan.
5. Gunakan evaluasi retrieval tambahan seperti MRR dan nDCG jika jumlah kasus relevan per query lebih dari satu.

In [5]:
summary_retrieval = retrieval_metrics[retrieval_metrics["query_id"] == "SUMMARY"]
summary_prediction = prediction_metrics[prediction_metrics["query_id"] == "SUMMARY"]

display(summary_retrieval)
display(summary_prediction)

,query_id,query,ground_truth_case_id,top_1_case_id,top_k_case_ids,hit_at_k,top_1_correct,precision_at_k,recall_at_k,f1_at_k
5,SUMMARY,,,,,1.0,1.0,0.2,1.0,0.333333


,query_id,query,expected_label,predicted_label,predicted_label_classifier,top_1_case_id,top_1_similarity,correct,precision_macro,recall_macro,f1_macro
5,SUMMARY,,,,,,,1.0,1.0,1.0,1.0


## Kesimpulan evaluasi

Model baseline TF-IDF dapat menemukan kasus pembanding dengan baik pada query evaluasi yang memiliki ground truth jelas. Tahap reuse juga dapat mengambil amar putusan dari kasus paling mirip. Namun, performa ini masih bergantung pada kualitas preprocessing, kelengkapan metadata, dan validitas ground truth. Karena itu, sistem lebih tepat disebut sebagai sistem pendukung analisis, bukan sistem pengganti pertimbangan hukum.